In [ ]:
# Dicas para executar notebooks no Google Colab:
# `pip install eegdash`
# Habilita visualizacao grafica inline no Jupyter notebook
%matplotlib inline

# Dividir EEG sem vazamento de sujeitos (*subject leakage*)

**Dificuldade 1-2** | **Tempo de execucao: 30s** | **Computacao: CPU**

Divisoes aleatorias de janelas em decodificadores de EEG inter-sujeito frequentemente apresentam acuracia de treino proxima de 99% e entram em colapso quando avaliadas em participantes retidos (*held-out*). O motivo nao e exotico: cada gravacao produz centenas de janelas sobrepostas originadas do mesmo cerebro, de modo que um embaralhamento uniforme espalha janelas de cada sujeito tanto no treino quanto no teste. O modelo passa a memorizar a assinatura individual de cada participante (frequencia cardiaca, amplitude de alfa, impedancia dos eletrodos) em vez da tarefa cognitiva que pretendemos decodificar.

Este tutorial demonstra a falha inicialmente em janelas sinteticas e, em seguida, reconstroi a divisao com as ferramentas do :mod:`eegdash.splits` e divisores inter-sujeito baseados em GroupKFold. A figura final compara as duas estrategias lado a lado: dados identicos, diferindo apenas na regra de divisao.

Brookshire et al. 2024 analisaram 81 artigos de aprendizado profundo aplicados a EEG e constataram vazamento de dados em aproximadamente metade deles. Cisotto & Chicco 2024 (Dica 9) classificam essa armadilha como a mais comum na avaliacao de EEG clinico; o benchmark MOABB :cite:`aristimunha2023transferstructure` adota o protocolo inter-sujeito estritamente.

.. sphinx_gallery_thumbnail_path = '_static/thumbs/plot_11_leakage_safe_split.png'

Palavras-chave: avaliacao, vazamento (*leakage*), divisao de dados


## Objetivos de aprendizagem

- Identificar o vazamento de participantes (*subject leakage*) como o principal modo de falha de divisoes aleatorias ingenuas em EEG.
- Construir uma validacao em 5 particoes livre de vazamento com ``get_splitter`` (``"cross_subject"``).
- Executar ``assert_no_leakage`` e interpretar a linha de auditoria JSON ``leakage_report`` emitida.
- Salvar um manifesto de divisao JSON com ``make_split_manifest`` e reproduzir uma particao com ``apply_split_manifest``.
- Visualizar o contraste entre um embaralhamento ingenuo e um GroupKFold inter-sujeito com a figura comparativa lado a lado.

## Requisitos

- Ter concluido :doc:`/generated/auto_examples/tutorials/10_core_workflow/plot_10_preprocess_and_window`.
- Cerca de 30 s em CPU. Sem uso de rede: a tabela de metadados e gerada em codigo.
- Recapitulacao conceitual: :doc:`/concepts/leakage_and_evaluation`.



Configuracao inicial. ``np.random.seed`` mantem o embaralhamento ingenuo e a ordem das particoes no manifesto reproduziveis (E3.21).



In [ ]:
# Importa modulos para serializacao JSON, sistema e controle de alertas
import json
import sys
import warnings
from pathlib import Path

# Importa bibliotecas para plotagem, manipulacao de arrays numericos e DataFrames
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Importa classes de contagem e rotinas do eegdash
import eegdash
from collections import Counter

# Importa divisores de validacao cruzada do MOABB e GroupKFold do scikit-learn
from moabb.evaluations.splitters import CrossSessionSplitter, CrossSubjectSplitter
from sklearn.model_selection import GroupKFold
from eegdash.viz import use_eegdash_style

# Aplica estilizacao grafica do EEGDash e silencia avisos futuros
use_eegdash_style()
warnings.simplefilter("ignore", category=FutureWarning)
# Fixa semente pseudoaleatoria para reprodutibilidade
SEED = 42
np.random.seed(SEED)
print(f"eegdash {eegdash.__version__}; numpy {np.__version__}")

## Por que o vazamento de sujeitos impacta o EEG mais intensamente do que outros dominios

Caracteristicas no nivel do participante dominam qualquer janela individual de EEG. Espessura craniana, posicionamento de eletrodos, amplitude basal do ritmo alfa, condutividade do couro cabeludo e batimentos cardiacos marcam os mesmos canais lidos pelo decodificador. Brookshire et al. 2024 quantificaram esse efeito em 81 artigos de aprendizado profundo em EEG clinico: quando os participantes figuravam simultaneamente em treino e teste, a acuracia media relatada atingia 0.83; quando avaliados com sujeitos retidos estritamente, as mesmas arquiteturas registraram media de 0.62. Metade dos estudos apresentava vazamento.

Outras modalidades frequentemente contornam essa questao. No ImageNet ha cerca de 1000 classes e mais de um milhao de imagens; no EEG ocorre o oposto: algumas dezenas de participantes fornecem centenas de janelas cada, todas carregando a assinatura individual daquele cerebro.

A solucao e estrutural: reter participantes completos, e nao janelas soltas. Agrupe cada janela pelo identificador do seu ``subject`` e utilize o :class:`sklearn.model_selection.GroupKFold` (ou o ``CrossSubjectSplitter`` do MOABB) para alocar cada participante a uma unica particao de teste.

## Valide seu resultado
- **Verificacao de vazamento:** execute a checagem de sobreposicao de conjuntos; nao deve haver intersecao de sujeitos entre treino e teste.
- **Relatorio de vazamento:** o relatorio JSON deve indicar zero participantes sobrepostos.
- **Diferenca de acuracia:** espere que uma divisao aleatoria ingenua supere a divisao segura em 10 a 30 pontos percentuais (o chamado "custo do vazamento").



## Etapa 1. Construir uma tabela de metadados de janelas para 12 participantes

Criamos uma tabela sintetica com 12 sujeitos, 2 sessoes e 8 janelas por sessao = 192 linhas, simulando o que e gerado na pratica a partir de dados reais como o ``ds002718``.



In [ ]:
# Define parametros da coorte simulada
N_SUBJECTS = 12
N_SESSIONS = 2
N_WINDOWS = 8
# Constroi lista de dicionarios simulando metadados de janelas BIDS
rows = [
    {
        "subject": f"sub-{s:02d}",
        "session": f"ses-{ses:02d}",
        "run": "run-01",
        "dataset": "ds-windowed-tutorial",
        "sample_id": f"sub-{s:02d}__ses-{ses:02d}__w{w:03d}",
        "target": int((s + w) % 2),
    }
    for s in range(1, N_SUBJECTS + 1)
    for ses in range(1, N_SESSIONS + 1)
    for w in range(N_WINDOWS)
]
raw_metadata = pd.DataFrame(rows)

# Em datasets do Braindecode, os metadados sao obtidos por BaseConcatDataset.get_metadata().
# Aqui usamos diretamente o DataFrame construido.
metadata = raw_metadata
y = metadata["target"].to_numpy()
# Exibe resumo da estrutura da tabela de metadados
pd.Series(
    {
        "rows": len(metadata),
        "subjects": metadata["subject"].nunique(),
        "sessions": metadata["session"].nunique(),
        "y dtype": str(y.dtype),
        "class 0 / class 1": (
            f"{int((metadata.target == 0).sum())} / {int((metadata.target == 1).sum())}"
        ),
    },
    name="value",
).to_frame()

## Etapa 2. Prever e executar da maneira ERRADA

**Preveja.** Se embaralharmos essas 192 janelas de modo puramente uniforme e colocarmos 20% em uma particao de teste, quantos sujeitos terminarao simultaneamente no treino E no teste? Escolha entre: 0, cerca de 5 ou todos os 12.

**Execute.** Um embaralhamento ingenuo no nivel de janelas: selecione 20% das janelas para teste.



In [ ]:
# Inicializa gerador pseudoaleatorio
rng = np.random.default_rng(SEED)
# Embaralha os indices das linhas de forma puramente aleatoria
shuffled = rng.permutation(len(metadata))
cut = int(0.8 * len(metadata))
# Divide em 80% treino e 20% teste sem considerar agrupamento por sujeito
naive_train = metadata.iloc[shuffled[:cut]]
naive_test = metadata.iloc[shuffled[cut:]]
# Calcula a intersecao de participantes presentes em ambos os conjuntos (vazamento)
leaked = sorted(set(naive_train["subject"]) & set(naive_test["subject"]))
naive_overlap = len(leaked)
# Exibe o resultado evidenciando o vazamento de sujeitos
pd.Series(
    {
        "train rows": len(naive_train),
        "test rows": len(naive_test),
        "subjects in train": naive_train["subject"].nunique(),
        "subjects in test": naive_test["subject"].nunique(),
        "subject_overlap": f"{naive_overlap} / {N_SUBJECTS}",
    },
    name="value",
).to_frame()

**Investigue.** Praticamente todos os participantes estao presentes nos dois lados da divisao. Um modelo ajustado nessa particao memoriza assinaturas individuais em vez de padroes generalizaveis da tarefa.



## Etapa 3. Construir um manifesto de divisao em 5 particoes livre de vazamento

**Execute.** O ``CrossSubjectSplitter`` com ``cv_class=GroupKFold`` garante que nenhum sujeito esteja simultaneamente no treino e no teste de qualquer particao.



In [ ]:
# Define quantidade de particoes para validacao cruzada
N_FOLDS = 5
# Instancia o divisor inter-sujeito utilizando GroupKFold com base na coluna subject
splitter = CrossSubjectSplitter(cv_class=GroupKFold, n_splits=N_FOLDS)
y = metadata["target"].to_numpy()
n_rows = len(metadata)
folds: list[tuple[np.ndarray, np.ndarray]] = []
# Itera pelas particoes gerando mascaras booleanas para treino e teste
for tr_idx, te_idx in splitter.split(y, metadata):
    tr_mask = np.zeros(n_rows, dtype=bool)
    tr_mask[tr_idx] = True
    te_mask = np.zeros(n_rows, dtype=bool)
    te_mask[te_idx] = True
    folds.append((tr_mask, te_mask))
# Exibe parametros da divisao estruturada
pd.Series(
    {
        "splitter_class": type(splitter).__name__,
        "n_folds": len(folds),
        "target": "target",
        "random_seed": SEED,
    },
    name="value",
).to_frame()

## Etapa 4. Provar a ausencia de vazamento de sujeitos e analisar a auditoria

Verificamos rigorosamente a ausencia de sobreposicao de participantes em todas as particoes.



In [ ]:
# Calcula a sobreposicao maxima de sujeitos entre treino e teste ao longo de todas as particoes
overlap = max(
    len(set(metadata.loc[tr, "subject"]) & set(metadata.loc[te, "subject"]))
    for tr, te in folds
)
# Emite a linha de relatorio de auditoria em formato JSON padronizado
sys.stdout.write(
    json.dumps({"leakage_report": {"overlap": int(overlap), "by": "subject"}}) + "\n"
)
sys.stdout.flush()
# Assercao rigorosa: o vazamento deve ser estritamente zero
assert overlap == 0, "Cross-subject split leaked!"

# Constroi a auditoria detalhada por particao
per_fold = []
for tr_mask, te_mask in folds:
    train = metadata.loc[tr_mask]
    test = metadata.loc[te_mask]
    per_fold.append(
        {
            "n_train": len(train),
            "n_test": len(test),
            "subjects_train": train["subject"].nunique(),
            "subjects_test": test["subject"].nunique(),
            "class_balance_train": dict(Counter(train["target"].dropna().tolist())),
            "class_balance_test": dict(Counter(test["target"].dropna().tolist())),
        }
    )
# Inspeciona o primeiro fold para checar o equilibrio de classes
fold0 = per_fold[0]
balance0 = fold0["class_balance_test"]
class_balance_ratio = max(balance0.values()) / (sum(balance0.values()) or 1)
pd.Series(
    {
        "fold": 0,
        "subjects_train": fold0["subjects_train"],
        "subjects_test": fold0["subjects_test"],
        "n_train": fold0["n_train"],
        "n_test": fold0["n_test"],
        "class_balance_test": dict(balance0),
        "class_balance_ratio": round(float(class_balance_ratio), 3),
    },
    name="value",
).to_frame()

## Etapa 5. Ler a tabela de auditoria por particao

Converter a auditoria em um DataFrame permite inspecionar visualmente o numero de sujeitos em cada particao e o equilibrio entre classes.



In [ ]:
# Converte a lista de auditoria das particoes em DataFrame
audit_df = pd.DataFrame(per_fold)
audit_df.insert(0, "fold", range(len(audit_df)))
# Exibe a tabela consolidada de auditoria por particao
audit_df[
    [
        "fold",
        "n_train",
        "n_test",
        "subjects_train",
        "subjects_test",
        "class_balance_train",
        "class_balance_test",
    ]
]

## Etapa 6. Materializar uma particao e persistir o manifesto

Salvamos o manifesto de divisao em um arquivo JSON padronizado, permitindo que qualquer colega ou pipeline automatizado reproduza exatamente os mesmos conjuntos de treino e teste.



In [ ]:
# Extrai as mascaras booleanas do primeiro fold
train_mask = folds[0][0]
test_mask = folds[0][1]
# Prepara o caminho do arquivo de manifesto JSON no cache
cache_dir = Path("./eegdash_cache")
cache_dir.mkdir(parents=True, exist_ok=True)
manifest_path = cache_dir / "plot_11_split_manifest.json"
# Constroi a estrutura com as amostras alocadas para treino e teste em cada fold
manifest_payload = {
    "splitter_class": type(splitter).__name__,
    "random_seed": SEED,
    "n_folds": len(folds),
    "target": "target",
    "folds": [
        {
            "train": metadata.loc[tr, "sample_id"].tolist(),
            "test": metadata.loc[te, "sample_id"].tolist(),
        }
        for tr, te in folds
    ],
}
# Grava o manifesto em disco em formato JSON
manifest_path.write_text(
    json.dumps(manifest_payload, sort_keys=True, default=str), encoding="utf-8"
)
# Exibe resumo da gravacao do manifesto
pd.Series(
    {
        "train_mask sum": int(train_mask.sum()),
        "test_mask sum": int(test_mask.sum()),
        "manifest bytes": manifest_path.stat().st_size,
        "manifest path": str(manifest_path),
    },
    name="value",
).to_frame()

## Resultado

A divisao ingenua vazou 11 de 12 participantes entre treino e teste; o manifesto inter-sujeito emite ``{"leakage_report": {"overlap": 0, "by": "subject"}}`` e distribui os 12 sujeitos de forma estanque e balanceada entre as 5 particoes.



In [ ]:
# Imprime o resumo final dos invariantes de integridade da validacao
print(
    "Final invariants:",
    json.dumps(
        {
            "n_subjects_total": int(metadata["subject"].nunique()),
            "n_folds": int(len(folds)),
            "subject_overlap": int(overlap),
            "naive_random_split_overlap": int(naive_overlap),
            "class_balance_ratio_fold0": round(float(class_balance_ratio), 3),
        }
    ),
)

## Um erro comum e como se recuperar

Dois deslizes comuns ocorrem neste processo: utilizar divisores incorretos (como divisores intra-sujeito para perguntas inter-sujeito) ou invocar um ``train_test_split`` puro do scikit-learn sem passar o argumento ``groups``, o que espalha janelas do mesmo participante em ambos os conjuntos.



In [ ]:
# Caso 1: Divisor intra-sujeito compartilha sujeitos intencionalmente, sendo inadequado para cross-subject
try:
    from moabb.evaluations.splitters import WithinSubjectSplitter

    bad = WithinSubjectSplitter(n_folds=N_FOLDS, random_state=SEED, shuffle=True)
    bad_folds = list(bad.split(y, metadata))
    bad_overlap_within = max(
        len(set(metadata.iloc[tr]["subject"]) & set(metadata.iloc[te]["subject"]))
        for tr, te in bad_folds
    )
    if bad_overlap_within > 0:
        raise ValueError(
            f"WithinSubjectSplitter shares {bad_overlap_within} subjects "
            "across train/test of every fold (expected — wrong splitter)"
        )
except ValueError as exc:
    print(f"Caught ValueError: {exc}")
    fixed = CrossSubjectSplitter(cv_class=GroupKFold, n_splits=N_FOLDS)
    print(f"Recovery: CrossSubjectSplitter -> {type(fixed).__name__}")

# Caso 2: train_test_split puro em janelas vaza participantes silenciosamente
from sklearn.model_selection import train_test_split

bad_train, bad_test = train_test_split(metadata, test_size=0.2, random_state=SEED)
bad_overlap = len(set(bad_train["subject"]) & set(bad_test["subject"]))
print(
    f"train_test_split(...) leaks {bad_overlap}/{N_SUBJECTS} subjects; "
    f"assert_no_leakage would raise LeakageError."
)

## Modifique. Experimente uma divisao baseada em sessoes

Substitua a regra para ``CrossSessionSplitter`` com divisao agrupada por ``session``.



In [ ]:
# Instancia divisor agrupado por sessao
session_splitter = CrossSessionSplitter(cv_class=GroupKFold, n_splits=2)
session_folds: list[tuple[np.ndarray, np.ndarray]] = []
# Divide os dados garantindo sessoes disjuntas
for tr_idx, te_idx in session_splitter.split(y, metadata):
    tr_mask = np.zeros(n_rows, dtype=bool)
    tr_mask[tr_idx] = True
    te_mask = np.zeros(n_rows, dtype=bool)
    te_mask[te_idx] = True
    session_folds.append((tr_mask, te_mask))
# Verifica sobreposicao de sessoes entre treino e teste
session_overlap = max(
    len(set(metadata.loc[tr, "session"]) & set(metadata.loc[te, "session"]))
    for tr, te in session_folds
)
print(f"cross_session overlap: {session_overlap}")

## Figura comparativa. Ingenua vs. Inter-Sujeito lado a lado

A rotina grafica ilustra a matriz de alocacao: no metodo ingenuo, todo sujeito e fatiado entre treino e teste em cada particao; na abordagem segura, cada participante e teste em exatamente uma particao.



In [ ]:
# Importa funcao auxiliar para plotagem das matrizes de alocacao
from _leakage_figure import draw_leakage_figure

# Seleciona ate 10 sujeitos para visualizacao grafica clara
subjects_for_fig = sorted(metadata["subject"].unique())[:10]
n_subj_fig = len(subjects_for_fig)
n_folds_fig = 5

# Matriz ingenua: valor 2 representa participante fatiado entre treino e teste
naive_assignment = np.full((n_subj_fig, n_folds_fig), 2, dtype=int)

# Matriz segura: valor 1 representa teste estrito; 0 representa treino estrito
safe_assignment = np.zeros((n_subj_fig, n_folds_fig), dtype=int)
for fold_index in range(min(n_folds_fig, len(folds))):
    test_mask = folds[fold_index][1]
    test_subjects = set(metadata.loc[test_mask, "subject"].unique())
    for row_idx, subject_id in enumerate(subjects_for_fig):
        if subject_id in test_subjects:
            safe_assignment[row_idx, fold_index] = 1

# Desenha e exibe a figura comparativa
fig = draw_leakage_figure(
    naive_assignment=naive_assignment,
    safe_assignment=safe_assignment,
    subjects=subjects_for_fig,
    n_windows_per_subject=N_SESSIONS * N_WINDOWS,
    plot_id="plot_11",
)
plt.show()

## Alternativa rapida: GroupShuffleSplit para particao unica de teste

Para uma divisao unica de treino/teste sem validacao cruzada completa, :class:`sklearn.model_selection.GroupShuffleSplit` baseado em ``subject`` fornece um par disjunto imediato.



In [ ]:
# Importa GroupShuffleSplit do scikit-learn
from sklearn.model_selection import GroupShuffleSplit

# Divide os dados em um unico conjunto de treino e teste respeitando os grupos de sujeitos
gss = GroupShuffleSplit(n_splits=1, test_size=0.4, random_state=SEED)
quick_tr_idx, quick_te_idx = next(gss.split(metadata, y, groups=metadata["subject"]))
quick_train = np.zeros(n_rows, dtype=bool)
quick_train[quick_tr_idx] = True
quick_test = np.zeros(n_rows, dtype=bool)
quick_test[quick_te_idx] = True
# Imprime resumo da divisao rapida
print(
    f"GroupShuffleSplit: train={int(quick_train.sum())} rows, "
    f"test={int(quick_test.sum())} rows | "
    f"test subjects={sorted(metadata.loc[quick_test, 'subject'].unique().tolist())}"
)
# Exibe contagens de linhas de cada particao do K-Fold principal
fold_sizes = [(int(tr.sum()), int(te.sum())) for tr, te in folds]
pd.DataFrame(fold_sizes, columns=["n_train_rows", "n_test_rows"]).head()

## Conclusao e referencias

Divisoes orientadas ao participante nao sao uma preferencia estetica em EEG; sao o unico protocolo que produz numeros comparaveis entre centros e artigos. O proximo tutorial (:doc:`/generated/auto_examples/tutorials/10_core_workflow/plot_12_train_a_baseline`) treina um modelo de base sobre essas particoes auditadas.

